# Reciprocal Rank Fusion (RRF) Submission Ensemble

This notebook merges the predictions of the high-precision **LLM Cross-Encoder Reranker** and the high-recall **Dense Bi-Encoder Retriever** using **Reciprocal Rank Fusion (RRF)**. 

### 🧭 Why Ensemble?
*   **Dense Retriever (Stage-1)** is excellent at capturing global semantic matchings, basic text concepts, and simple lexical alignments.
*   **LLM Reranker (Stage-2)** performs deep instruction-following, logical reasoning, and close context examination, but might occasionally miss basic matches if they were filtered too aggressively.
*   **RRF** blends both predictions, taking advantage of both strengths to generate highly robust, competitive Kaggle submissions!

In [1]:
# === SYSTEM SETUP & PATH RESOLUTION ===
import sys
import pandas as pd
from pathlib import Path

# Setup paths relative to parent directory (adds parent dir to Python path for src imports)
base_dir = Path("..").resolve()
sys.path.append(str(base_dir))
sub_dir = base_dir / "outputs/submissions"

# === MODULE IMPORTS ===
from src.data.loader import load_jsonl
from src.evaluation.metrics import calculate_recall_at_k, calculate_mrr_at_k, calculate_hit_rate_at_k

# === UTILITY FUNCTIONS ===
def compute_rrf_score(ranks, weights, k=10):
    """
    Computes Reciprocal Rank Fusion score.
    ranks: dict mapping model_name -> rank (1-based). If missing, treated as rank infinity (score = 0).
    weights: dict mapping model_name -> weight.
    """
    score = 0.0
    for model_name, rank in ranks.items():
        if rank is not None:
            weight = weights.get(model_name, 1.0)
            score += weight / (k + rank)
    return score


In [2]:
print("=== 🤝 INITIALIZING 3-WAY RECIPROCAL RANK FUSION ENSEMBLE ===")

# 1. Define input files and weights dynamically
K_CONSTANT = 10  # Tuned RRF constant for top-5 ranking lists

INPUT_CONFIG = [
    {"path": base_dir / "outputs/submissions/submission_none_hybrid(qwen3-embedding-8b + bge-m3)_qwen2.5-32b-instruct_none.csv", "weight": 0.75},
    {"path": base_dir / "outputs/submissions/submission_none_hybrid(qwen3-embedding-8b + bge-m3)_none_none.csv", "weight": 0.15},
    {"path": base_dir / "outputs/submissions/submission_none_bm25_none_none.csv", "weight": 0.10},
]

valid_configs = []
print(f"📂 Fusing predictions from:")
for cfg in INPUT_CONFIG:
    if cfg["path"].exists():
        valid_configs.append(cfg)
        print(f"  - [Weight {cfg['weight']:.2f}]: {cfg['path'].name}")
    else:
        print(f"  - ❌ NOT FOUND: {cfg['path'].name}")

if len(valid_configs) < 2:
    raise ValueError("Need at least 2 valid submission files for ensembling!")


=== 🤝 INITIALIZING 3-WAY RECIPROCAL RANK FUSION ENSEMBLE ===
📂 Fusing predictions from:
  - [Weight 0.75]: submission_none_hybrid(qwen3-embedding-8b + bge-m3)_qwen2.5-32b-instruct_none.csv
  - [Weight 0.15]: submission_none_hybrid(qwen3-embedding-8b + bge-m3)_none_none.csv
  - [Weight 0.10]: submission_none_bm25_none_none.csv


In [3]:
# 2. Load and verify submissions
dataframes = []
for cfg in valid_configs:
    df = pd.read_csv(cfg["path"])
    df = df.sort_values("q_id").reset_index(drop=True)
    dataframes.append(df)

# Ensure all lengths match
base_len = len(dataframes[0])
assert all(len(df) == base_len for df in dataframes), "Length mismatch between submissions!"


In [4]:
# 3. Execute Dynamic RRF
fused_predictions = []

for idx in range(base_len):
    q_id = dataframes[0].iloc[idx]["q_id"]
    item_scores = {}
    
    for df_idx, cfg in enumerate(valid_configs):
        items_str = str(dataframes[df_idx].iloc[idx]["gold_quotes"])
        if items_str == 'nan':
            items_str = ''
        items = items_str.split()
        weight = cfg["weight"]
        for rank, item in enumerate(items):
            if item not in item_scores:
                item_scores[item] = 0.0
            item_scores[item] += weight / (K_CONSTANT + (rank + 1))
            
    # Sort candidates by fused score descending
    sorted_candidates = sorted(item_scores.keys(), key=lambda x: item_scores[x], reverse=True)
    top_5 = sorted_candidates[:5]
    
    fused_predictions.append({
        "q_id": q_id,
        "gold_quotes": " ".join(top_5)
    })

# 4. Determine output directory and filename based on inputs
is_val = any('validation' in cfg["path"].name or 'val_' in cfg["path"].name for cfg in valid_configs)

# Automatically route to the correct folder!
out_folder = base_dir / ("outputs/validation" if is_val else "outputs/submissions")
prefix = "val" if is_val else "submission"

# Sort configs so the one with the reranker (Stage-2) is treated as the primary name
sorted_configs = sorted(
    valid_configs, 
    key=lambda c: any(reranker in c["path"].name for reranker in ["qwen2.5", "qwen3-reranker", "bge-reranker"]),
    reverse=True
)
primary_path = sorted_configs[0]["path"]

# Extract the core experiment name (stripping 'val_' and '_none')
core_name = primary_path.stem
for pfx in ["val_", "validation_", "submission_"]:
    if core_name.startswith(pfx):
        core_name = core_name[len(pfx):]
if core_name.endswith("_none"):
    core_name = core_name[:-5]

# Build the final absolute path
output_file = out_folder / f"{prefix}_{core_name}_3way-rrf.csv"

# Save the fused dataframe
df_fused = pd.DataFrame(fused_predictions)
df_fused.to_csv(output_file, index=False)

print(f"\n✅ ENSEMBLE SUCCESSFULLY COMPUTED AND SAVED TO:\n{output_file}")


✅ ENSEMBLE SUCCESSFULLY COMPUTED AND SAVED TO:
/data220_2/emmy/mlbio/hw3/outputs/submissions/submission_none_hybrid(qwen3-embedding-8b + bge-m3)_qwen2.5-32b-instruct_3way-rrf.csv


In [5]:
# 5. Print refinement statistics (compared to first input)
changed_count = 0
primary_df = dataframes[0]
for i in range(base_len):
    primary_quotes = str(primary_df.iloc[i]["gold_quotes"])
    if primary_quotes == 'nan': primary_quotes = ''
    fused_quotes = str(df_fused.iloc[i]["gold_quotes"])
    if primary_quotes != fused_quotes:
        changed_count += 1
        
percent = (changed_count / base_len) * 100
print(f"📊 Ensemble Statistics (vs primary input):")
print(f"  - Total refined samples: {changed_count} / {base_len} ({percent:.1f}%)")
print(f"  - These {changed_count} samples now contain blended items from all valid pipelines.")


📊 Ensemble Statistics (vs primary input):
  - Total refined samples: 1188 / 1798 (66.1%)
  - These 1188 samples now contain blended items from all valid pipelines.


In [6]:
print('\n✅ Kaggle Test Set Submission RRF Generated!')


✅ Kaggle Test Set Submission RRF Generated!
